# Milestone 5

Master the competition metric, optimise inference, and combine models for maximum performance. Introduce TTA.

Suggested Readings:
 - PyTorch – Softmax Function
 - Hugging Face – Transformer
 - Ensemble Learning

[Competition link](https://www.kaggle.com/competitions/smart-mcq-solver-challenge)

Setup (Run Before Attempting Questions) : 

Use the following two fine-tuned sequence classification checkpoints:
1. DeBERTa: microsoft/deberta-v3-small (fine-tuned checkpoint)
2. RoBERTa: roberta-base (fine-tuned checkpoint)
Label Mapping
The models output logits for five labels corresponding to the answer options:

|Label ID|Option|
|--- | --- |
|0|A|
|1|B|
|2|C|
|3|D|
|4|E|


In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
!pip install -q transformers datasets peft accelerate

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForMultipleChoice

train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')

label_to_index = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
index_to_label = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}

print(f"Dataset loaded successfully with {len(train)} rows.")

Dataset loaded successfully with 2000 rows.


Model Checkpoint Paths & Load Tokenizers

In [3]:
DEBERTA_MODEL_NAME = "microsoft/deberta-v3-small"
ROBERTA_MODEL_NAME = "roberta-base"

deberta_tokenizer = AutoTokenizer.from_pretrained(DEBERTA_MODEL_NAME)
roberta_tokenizer = AutoTokenizer.from_pretrained(ROBERTA_MODEL_NAME)

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Create reusable data formating & Tokenization function

In [4]:
def prepare_mc_inputs(df, tokenizer, max_length=128):
    all_input_ids = []
    all_attention_masks = []
    options = ['A', 'B', 'C', 'D', 'E']
    
    for _, row in df.iterrows():
        prompt = str(row['prompt'])
        choices = [prompt + " " + (tokenizer.sep_token or "[SEP]") + " " + str(row[opt]) for opt in options]
        
        tokenized = tokenizer(
            choices,
            padding="max_length",
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )
        
        all_input_ids.append(tokenized['input_ids'])
        all_attention_masks.append(tokenized['attention_mask'])
        
    input_ids = torch.stack(all_input_ids)           # Shape: [N, 5, max_length]
    attention_mask = torch.stack(all_attention_masks) # Shape: [N, 5, max_length]
    
    return input_ids, attention_mask

Load Model Weights and move to GPU

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

deberta_model = AutoModelForMultipleChoice.from_pretrained(DEBERTA_MODEL_NAME).to(device)
deberta_model.eval()

roberta_model = AutoModelForMultipleChoice.from_pretrained(ROBERTA_MODEL_NAME).to(device)
roberta_model.eval()

print(f"Models initialized and running on device: {device}")

pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForMultipleChoice LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias               

model.safetensors:   0%|          | 0.00/286M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForMultipleChoice LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
roberta.pooler.dense.weight     | MISSING    | 
roberta.pooler.dense.bias       | MISSING    | 
classifier.weight               | MISSING    | 
classifier.bias                 | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Models initialized and running on device: cuda


Load the fine-tuned DeBERTa and RoBERTa models.

For the prompt at row index 25, perform inference using each model independently and apply Softmax to obtain class probabilities.

Question 1:

Which answer option receives the highest probability from the DeBERTa model, and what is that probability?


(answer format : eg - A, probability of A)

In [6]:
row_25 = train.iloc[[25]]

input_ids_deb, attention_mask_deb = prepare_mc_inputs(row_25, deberta_tokenizer, max_length=128)
input_ids_deb = input_ids_deb.to(device)
attention_mask_deb = attention_mask_deb.to(device)

with torch.no_grad():
    deberta_outputs = deberta_model(input_ids=input_ids_deb, attention_mask=attention_mask_deb)

deberta_logits = deberta_outputs.logits  # Shape: [1, 5]
deberta_probs = F.softmax(deberta_logits, dim=-1).squeeze(0).cpu().numpy()

top_idx = np.argmax(deberta_probs)
top_option = index_to_label[top_idx]
top_prob = deberta_probs[top_idx]

print(f"DeBERTa Probabilities: {dict(zip(['A','B','C','D','E'], deberta_probs))}")
print(f"Result: {top_option}, {top_prob:.4f}")

DeBERTa Probabilities: {'A': np.float16(0.2034), 'B': np.float16(0.1979), 'C': np.float16(0.2024), 'D': np.float16(0.2025), 'E': np.float16(0.194)}
Result: A, 0.2034


Using the same sample (row index 25), average the class probabilities from both models.

Average Probability = \[P(DeBERTa) + P(RoBERTa)]/2

Question 2:

Which answer option receives the highest averaged probability after simple probability ensembling?

In [7]:
input_ids_rob, attention_mask_rob = prepare_mc_inputs(row_25, roberta_tokenizer, max_length=128)
input_ids_rob = input_ids_rob.to(device)
attention_mask_rob = attention_mask_rob.to(device)

with torch.no_grad():
    roberta_outputs = roberta_model(input_ids=input_ids_rob, attention_mask=attention_mask_rob)

roberta_logits = roberta_outputs.logits  # Shape: [1, 5]
roberta_probs = F.softmax(roberta_logits, dim=-1).squeeze(0).cpu().numpy()

averaged_probs = (deberta_probs + roberta_probs) / 2.0

top_avg_idx = np.argmax(averaged_probs)
top_avg_option = index_to_label[top_avg_idx]
top_avg_prob = averaged_probs[top_avg_idx]

options = ['A', 'B', 'C', 'D', 'E']
print("DeBERTa Probs :", dict(zip(options, np.round(deberta_probs, 4))))
print("RoBERTa Probs :", dict(zip(options, np.round(roberta_probs, 4))))
print("Averaged Probs:", dict(zip(options, np.round(averaged_probs, 4))))

print(f"\nHighest averaged option: {top_avg_option} (Probability: {top_avg_prob:.4f})")

DeBERTa Probs : {'A': np.float16(0.2034), 'B': np.float16(0.1979), 'C': np.float16(0.2024), 'D': np.float16(0.2025), 'E': np.float16(0.194)}
RoBERTa Probs : {'A': np.float32(0.1996), 'B': np.float32(0.2), 'C': np.float32(0.2), 'D': np.float32(0.1992), 'E': np.float32(0.2011)}
Averaged Probs: {'A': np.float32(0.2015), 'B': np.float32(0.1989), 'C': np.float32(0.2012), 'D': np.float32(0.2009), 'E': np.float32(0.1975)}

Highest averaged option: A (Probability: 0.2015)


Apply weighted probability averaging after Softmax using the following weights:
- DeBERTa: 0.70
- RoBERTa: 0.30
  
Compute:
- P(final) = \[0.7 × P(DeBERTa)] + \[0.3 × P(RoBERTa)]
  
Question 3:

Which answer option is ranked first after weighted ensembling?

In [8]:
w_deberta = 0.70
w_roberta = 0.30

weighted_probs = (w_deberta * deberta_probs) + (w_roberta * roberta_probs)

top_weighted_idx = np.argmax(weighted_probs)
top_weighted_option = index_to_label[top_weighted_idx]
top_weighted_prob = weighted_probs[top_weighted_idx]

options = ['A', 'B', 'C', 'D', 'E']
print("Weighted Probs:", dict(zip(options, np.round(weighted_probs, 4))))
print(f"\nHighest weighted option: {top_weighted_option} (Probability: {top_weighted_prob:.4f})")

Weighted Probs: {'A': np.float32(0.2023), 'B': np.float32(0.1986), 'C': np.float32(0.2017), 'D': np.float32(0.2016), 'E': np.float32(0.1962)}

Highest weighted option: A (Probability: 0.2023)


Using the weighted ensemble probabilities from Q3, rank all five answer options.

Write the final prediction exactly in Kaggle submission format.

Question 4:

What is the Top-3 prediction string for row index 25?

Example : C A E

In [9]:
sorted_indices = np.argsort(weighted_probs)[::-1]

top_3_options = [index_to_label[idx] for idx in sorted_indices[:3]]

top_3_string = " ".join(top_3_options)

print(f"Full Ranked Options: {[index_to_label[idx] for idx in sorted_indices]}")
print(f"Top-3 Prediction String: {top_3_string}")

Full Ranked Options: ['A', 'C', 'D', 'B', 'E']
Top-3 Prediction String: A C D


Run the weighted ensemble pipeline on every row of ***test.csv***.

Save the predictions in a file named ***submission.csv*** using the required Kaggle format:

id,prediction

where the prediction column contains the Top-3 ranked options separated by spaces.

Question 5:

Exactly how many prediction rows are present in the generated file (excluding the header)?

In [10]:
test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

submission_rows = []

for idx, row in test_df.iterrows():
    row_df = test_df.iloc[[idx]]
    row_id = row['id']
    
    ids_deb, mask_deb = prepare_mc_inputs(row_df, deberta_tokenizer, max_length=128)
    ids_rob, mask_rob = prepare_mc_inputs(row_df, roberta_tokenizer, max_length=128)
    
    ids_deb, mask_deb = ids_deb.to(device), mask_deb.to(device)
    ids_rob, mask_rob = ids_rob.to(device), mask_rob.to(device)
    
    with torch.no_grad():
        out_deb = deberta_model(input_ids=ids_deb, attention_mask=mask_deb)
        out_rob = roberta_model(input_ids=ids_rob, attention_mask=mask_rob)
        
    p_deb = F.softmax(out_deb.logits, dim=-1).squeeze(0).cpu().numpy()
    p_rob = F.softmax(out_rob.logits, dim=-1).squeeze(0).cpu().numpy()
    
    p_weighted = (0.70 * p_deb) + (0.30 * p_rob)
    
    sorted_indices = np.argsort(p_weighted)[::-1]
    top_3_str = " ".join([index_to_label[i] for i in sorted_indices[:3]])
    
    submission_rows.append({'id': row_id, 'prediction': top_3_str})

submission_df = pd.DataFrame(submission_rows)
submission_df.to_csv('submission.csv', index=False)

print(f"Submission saved successfully to submission.csv.")
print(f"Number of prediction rows (excluding header): {len(submission_df)}")

Submission saved successfully to submission.csv.
Number of prediction rows (excluding header): 500


For the first 50 rows of test.csv, create two versions of every prompt:
1. Original prompt
2. Instruction-augmented prompt by prepending: "Answer the following multiple-choice question carefully:"
   
Run inference using DeBERTa on both versions.

Average the predicted probabilities from both passes.

Question 6:

How many of the first 50 rows produce a different Top-1 prediction after applying Test-Time Augmentation?

In [12]:
test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv').iloc[:50]

different_top1_count = 0
options = ['A', 'B', 'C', 'D', 'E']

for idx, row in test_df.iterrows():
    row_df = test_df.iloc[[idx]]
    prompt_orig = str(row['prompt'])
    
    ids_orig, mask_orig = prepare_mc_inputs(row_df, deberta_tokenizer, max_length=128)
    ids_orig, mask_orig = ids_orig.to(device), mask_orig.to(device)
    
    augmented_prompt = "Answer the following multiple-choice question carefully: " + prompt_orig
    aug_choices = [augmented_prompt + " " + (deberta_tokenizer.sep_token or "[SEP]") + " " + str(row[opt]) for opt in options]
    
    tokenized_aug = deberta_tokenizer(
        aug_choices,
        padding="max_length",
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )
    ids_aug = tokenized_aug['input_ids'].unsqueeze(0).to(device)
    mask_aug = tokenized_aug['attention_mask'].unsqueeze(0).to(device)
    
    with torch.no_grad():
        out_orig = deberta_model(input_ids=ids_orig, attention_mask=mask_orig)
        out_aug = deberta_model(input_ids=ids_aug, attention_mask=mask_aug)
        
    p_orig = F.softmax(out_orig.logits, dim=-1).squeeze(0).cpu().numpy()
    p_aug = F.softmax(out_aug.logits, dim=-1).squeeze(0).cpu().numpy()
    
    top1_orig_idx = np.argmax(p_orig)
    
    p_tta_avg = (p_orig + p_aug) / 2.0
    top1_tta_idx = np.argmax(p_tta_avg)
    
    if top1_orig_idx != top1_tta_idx:
        different_top1_count += 1

print(f"Number of rows with a different Top-1 prediction after TTA: {different_top1_count}")

Number of rows with a different Top-1 prediction after TTA: 12


Process the first 100 rows of test.csv. And compare the Top-1 prediction from:
1. DeBERTa
2. Weighted Ensemble
   
Question 7:

How many rows have different Top-1 predictions?

In [14]:
test_100 = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv').iloc[:100]

different_top1_count = 0

for idx, row in test_100.iterrows():
    row_df = test_100.iloc[[idx]]
    
    ids_deb, mask_deb = prepare_mc_inputs(row_df, deberta_tokenizer, max_length=128)
    ids_rob, mask_rob = prepare_mc_inputs(row_df, roberta_tokenizer, max_length=128)
    
    ids_deb, mask_deb = ids_deb.to(device), mask_deb.to(device)
    ids_rob, mask_rob = ids_rob.to(device), mask_rob.to(device)
    
    with torch.no_grad():
        out_deb = deberta_model(input_ids=ids_deb, attention_mask=mask_deb)
        out_rob = roberta_model(input_ids=ids_rob, attention_mask=mask_rob)
        
    p_deb = F.softmax(out_deb.logits, dim=-1).squeeze(0).cpu().numpy()
    p_rob = F.softmax(out_rob.logits, dim=-1).squeeze(0).cpu().numpy()
    
    top1_deberta_idx = np.argmax(p_deb)
    
    p_weighted = (0.70 * p_deb) + (0.30 * p_rob)
    top1_ensemble_idx = np.argmax(p_weighted)
    
    if top1_deberta_idx != top1_ensemble_idx:
        different_top1_count += 1

print(f"Number of rows with different Top-1 predictions: {different_top1_count}")

Number of rows with different Top-1 predictions: 5


For the first 100 rows of test.csv, record the highest class probability (confidence) predicted by:
1. DeBERTa
2. Weighted Ensemble
   
For every row, compute:

Confidence Gain = Ensemble Confidence−DeBERTa Confidence

Question 8:
How many rows have a positive confidence gain (greater than 0)?

In [15]:
test_100 = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv').iloc[:100]

positive_gain_count = 0

for idx, row in test_100.iterrows():
    row_df = test_100.iloc[[idx]]
    
    ids_deb, mask_deb = prepare_mc_inputs(row_df, deberta_tokenizer, max_length=128)
    ids_rob, mask_rob = prepare_mc_inputs(row_df, roberta_tokenizer, max_length=128)
    
    ids_deb, mask_deb = ids_deb.to(device), mask_deb.to(device)
    ids_rob, mask_rob = ids_rob.to(device), mask_rob.to(device)
    
    with torch.no_grad():
        out_deb = deberta_model(input_ids=ids_deb, attention_mask=mask_deb)
        out_rob = roberta_model(input_ids=ids_rob, attention_mask=mask_rob)
        
    p_deb = F.softmax(out_deb.logits, dim=-1).squeeze(0).cpu().numpy()
    p_rob = F.softmax(out_rob.logits, dim=-1).squeeze(0).cpu().numpy()
    
    confidence_deberta = np.max(p_deb)
    
    p_weighted = (0.70 * p_deb) + (0.30 * p_rob)
    confidence_ensemble = np.max(p_weighted)
    
    confidence_gain = confidence_ensemble - confidence_deberta
    
    if confidence_gain > 0:
        positive_gain_count += 1

print(f"Number of rows with a positive confidence gain (> 0): {positive_gain_count}")

Number of rows with a positive confidence gain (> 0): 1


For the first 100 rows of test.csv, compare the Top-3 prediction strings generated by:
1. DeBERTa alone
2. Weighted Ensemble
   
Question 9:

How many rows have at least one change in their ordered Top-3 ranking after ensembling?

Examples:

A C D vs. A D C

In [16]:
test_100 = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv').iloc[:100]

ranking_change_count = 0

for idx, row in test_100.iterrows():
    row_df = test_100.iloc[[idx]]
    
    ids_deb, mask_deb = prepare_mc_inputs(row_df, deberta_tokenizer, max_length=128)
    ids_rob, mask_rob = prepare_mc_inputs(row_df, roberta_tokenizer, max_length=128)
    
    ids_deb, mask_deb = ids_deb.to(device), mask_deb.to(device)
    ids_rob, mask_rob = ids_rob.to(device), mask_rob.to(device)
    
    with torch.no_grad():
        out_deb = deberta_model(input_ids=ids_deb, attention_mask=mask_deb)
        out_rob = roberta_model(input_ids=ids_rob, attention_mask=mask_rob)
        
    p_deb = F.softmax(out_deb.logits, dim=-1).squeeze(0).cpu().numpy()
    p_rob = F.softmax(out_rob.logits, dim=-1).squeeze(0).cpu().numpy()
    
    sorted_deb_indices = np.argsort(p_deb)[::-1]
    top3_deberta_str = " ".join([index_to_label[i] for i in sorted_deb_indices[:3]])
    
    p_weighted = (0.70 * p_deb) + (0.30 * p_rob)
    sorted_ens_indices = np.argsort(p_weighted)[::-1]
    top3_ensemble_str = " ".join([index_to_label[i] for i in sorted_ens_indices[:3]])
    
    if top3_deberta_str != top3_ensemble_str:
        ranking_change_count += 1

print(f"Number of rows with at least one change in their ordered Top-3 ranking: {ranking_change_count}")

Number of rows with at least one change in their ordered Top-3 ranking: 23


Using the Top-3 predictions generated by your weighted ensemble for the first 100 validation samples, compute the MAP@3 score.

Question 10:

What is the final MAP@3 score? 

(Round to 4 decimal places.)

In [17]:
val_100 = train.iloc[:100]

def calculate_map3(predictions, ground_truths):
    scores = []
    for pred, actual in zip(predictions, ground_truths):
        score = 0.0
        for rank, item in enumerate(pred[:3]):
            if item == actual:
                score = 1.0 / (rank + 1)
                break
        scores.append(score)
    return np.mean(scores)

top3_predictions = []
actual_labels = []

for idx, row in val_100.iterrows():
    row_df = val_100.iloc[[idx]]
    
    actual_labels.append(str(row['answer']))
    
    ids_deb, mask_deb = prepare_mc_inputs(row_df, deberta_tokenizer, max_length=128)
    ids_rob, mask_rob = prepare_mc_inputs(row_df, roberta_tokenizer, max_length=128)
    
    ids_deb, mask_deb = ids_deb.to(device), mask_deb.to(device)
    ids_rob, mask_rob = ids_rob.to(device), mask_rob.to(device)
    
    with torch.no_grad():
        out_deb = deberta_model(input_ids=ids_deb, attention_mask=mask_deb)
        out_rob = roberta_model(input_ids=ids_rob, attention_mask=mask_rob)
        
    p_deb = F.softmax(out_deb.logits, dim=-1).squeeze(0).cpu().numpy()
    p_rob = F.softmax(out_rob.logits, dim=-1).squeeze(0).cpu().numpy()
    
    p_weighted = (0.70 * p_deb) + (0.30 * p_rob)
    
    sorted_indices = np.argsort(p_weighted)[::-1]
    top_3_options = [index_to_label[i] for i in sorted_indices[:3]]
    top3_predictions.append(top_3_options)

final_map3 = calculate_map3(top3_predictions, actual_labels)

print(f"Final MAP@3 Score: {final_map3:.4f}")

Final MAP@3 Score: 0.3083
